# Análisis de cambio semánticos (generar datos para análisis)

In [1]:
# Importar librería
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import pickle
import os
import re
import random as rn
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import bootstrap
import matplotlib.pyplot as plt
import itertools

from gensim.models import Word2Vec
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
tqdm.pandas()

from ast import literal_eval

In [2]:
# Configurar
load_dotenv() # Cargar las variables de entorno del archivo .env
BASE_DIR =  os.getenv("DIR_BASE")
RESULTADOS_DIR = os.getenv("DIR_DATOS_PROCESADOS") # Acceder a las variables de entorno modelos_swmwosge_1008211100
modelo_dir =  RESULTADOS_DIR+ '/archivos_out/modelos_swmwosge_5051100'
pd.set_option('display.max_colwidth', None)

Se selecciona para analizar el modelo de procrustes

In [3]:
basename = 'procrustes'
iteracion = 50
tam_vector = 50

In [4]:
# estabilidad_procrustes_iter10_tam50
selected_topics_df = pd.read_csv(
    modelo_dir+'/estabilidad_procrustes/estabilidad_'+basename+'_iter'+str(iteracion)+'_tam'+str(tam_vector)+'.csv',
    converters={'par_periodo': literal_eval})

selected_topics_df.head(5)

,iteracion,par_periodo,palabra,similaridad_semantica,cantidad_palabras_comun,top10_vecindad_t1,top10_vecindad_t2
0,6,"(2014, 2019)",accion,0.026791,461,"[('judicial', 0.6770266890525818), ('comunitario', 0.6661193370819092), ('julio', 0.6481040120124817), ('mental', 0.6264558434486389), ('nacimiento', 0.6204757690429688), ('valor', 0.5921574831008911), ('primario', 0.579254150390625), ('desarrollo', 0.5711088180541992), ('premio', 0.5692583322525024), ('psicologico', 0.5626101493835449)]","[('epidemiologico', 0.6849518418312073), ('promover', 0.6302334666252136), ('inter', 0.6208462119102478), ('organo', 0.6154682040214539), ('celula', 0.6115739941596985), ('tratamiento', 0.6076741218566895), ('nivel', 0.5620438456535339), ('donacion', 0.5555312633514404), ('padecer', 0.554180383682251), ('investigacion', 0.538840115070343)]"
1,40,"(2014, 2019)",dispositivo,0.067824,461,"[('nocivo', 0.7619382739067078), ('similar', 0.6900781989097595), ('electronico', 0.6778493523597717), ('dato', 0.6534647941589355), ('digital', 0.6030073761940002), ('efecto', 0.5861523151397705), ('necesario', 0.585832417011261), ('medicamentosa', 0.5726563930511475), ('transgenico', 0.5610784292221069), ('leches', 0.5532217621803284)]","[('tecnologico', 0.6621898412704468), ('muerte_subita', 0.6154015064239502), ('minimo', 0.5942965149879456), ('desfibrilador', 0.5798337459564209), ('oftalmologico', 0.565621018409729), ('concurrencia', 0.5628604888916016), ('comprendido', 0.5361419320106506), ('lugar', 0.5149423480033875), ('calidad', 0.513451099395752), ('examen', 0.5114758014678955)]"
2,35,"(2009, 2014)",basico,0.071167,632,"[('laboratorio', 0.6807141900062561), ('menor', 0.6475790143013), ('fincini', 0.6377453804016113), ('ninez', 0.631625235080719), ('ganancia', 0.6304410696029663), ('ingreso', 0.6263360977172852), ('utilizacion', 0.6157370805740356), ('calidad', 0.6059296727180481), ('distribucion', 0.6013383269309998), ('padezcar', 0.5793224573135376)]","[('reduccion', 0.7038561701774597), ('objeto', 0.7004052400588989), ('precio', 0.6229988932609558), ('nacido', 0.6214845776557922), ('recien', 0.6185654401779175), ('equipo', 0.6037498712539673), ('primaria', 0.5589615702629089), ('situacion', 0.5472093224525452), ('celiaco', 0.538116455078125), ('argentina', 0.5338548421859741)]"
3,44,"(2009, 2014)",basico,0.080370,632,"[('laboratorio', 0.6913539171218872), ('ninez', 0.6830536127090454), ('ingreso', 0.6494584679603577), ('ganancia', 0.6301590204238892), ('fincini', 0.6233099699020386), ('tabaco', 0.617645263671875), ('utilizacion', 0.6065181493759155), ('menor', 0.5954523682594299), ('distribucion', 0.5937909483909607), ('limitacion', 0.5896505117416382)]","[('reduccion', 0.6890735030174255), ('objeto', 0.679164707660675), ('precio', 0.652248203754425), ('recien', 0.652148425579071), ('nacido', 0.6508929133415222), ('equipo', 0.6109211444854736), ('primaria', 0.5782000422477722), ('bis', 0.5420730710029602), ('situacion', 0.522652804851532), ('geriatrico', 0.5151534080505371)]"
4,33,"(2014, 2019)",adecuado,0.090077,461,"[('rotulacion', 0.7852411866188049), ('especialidad', 0.6834930181503296), ('porcentaje', 0.6528801321983337), ('camara', 0.6461610198020935), ('gasificado', 0.6428849101066589), ('frio', 0.6421188712120056), ('jugo', 0.6405826807022095), ('cadena', 0.640575647354126), ('alcohol', 0.6377267241477966), ('medicinal', 0.6370911598205566)]","[('manejo', 0.6844812631607056), ('geriatrico', 0.6576078534126282), ('tumor', 0.6037538051605225), ('hijo', 0.5944344997406006), ('agente', 0.5880053043365479), ('situacion', 0.5502997636795044), ('procreacion', 0.5487896800041199), ('paliativo', 0.5447098016738892), ('modificatoria', 0.5377705693244934), ('funcion', 0.5327803492546082)]"


In [5]:
selected_topics_df.describe() # Estadística

,iteracion,similaridad_semantica,cantidad_palabras_comun
count,54650.000000,54650.000000,54650.000000
mean,24.500000,0.640435,559.876487
std,14.431002,0.139921,84.447914
min,0.000000,0.026791,461.000000
25%,12.000000,0.551986,461.000000
50%,24.500000,0.651548,632.000000
75%,37.000000,0.740335,632.000000
max,49.000000,0.945866,632.000000


Se realizaron 50 ejecuciones independientes del enfoque Procrustes, cada una con una semilla aleatoria distinta. En cada iteración, se seleccionaron las 250 palabras con mayor cambio semántico entre los pares de períodos comparados, considerando únicamente aquellas con al menos 5 ocurrencias en alguno de los dos períodos de cinco años.

Posteriormente, se calculó la similaridad semantica promedio (medida de estabilidad del modelo) para cada palabra identificada en cada par de periodos comparados. Este valor fue estimado junto con sus respectivos intervalos de confianza del 95%, utilizando el método bootstrap.

In [6]:
n = 250
at_least_in_any_decade = 5
pares_periodo = [ (2009, 2014),(2014, 2019)]
topn_periodo = {}

for par_periodo in pares_periodo:
    df =  selected_topics_df[
        selected_topics_df['par_periodo'] == par_periodo
    ].copy()
    
    
    frecPer1 = pd.read_csv(RESULTADOS_DIR+'/archivos_out/frec_para_datos_limpios_por_desplaz_semantico_anios5_'+str(par_periodo[0])+'.csv')
    frecPer2 = pd.read_csv(RESULTADOS_DIR+'/archivos_out/frec_para_datos_limpios_por_desplaz_semantico_anios5_'+str(par_periodo[1])+'.csv')
    frecPer1.columns = ['palabra', 'frec1', 'porc1']
    frecPer2.columns = ['palabra', 'frec2', 'porc2']
    #print(frecPer2.shape)
    topn_all_df = []

    for iter in range(iteracion):
        #print(iter)
        df_topn = df[(df.iteracion==iter)]
        #controlar frec
        #display(df_topn)
        df_topn = df_topn.merge(frecPer1, how='left', on='palabra')
        df_topn = df_topn.merge(frecPer2, how='left', on='palabra')
        df_topn['max_freq_of_any_decade'] = df_topn[['frec1', 'frec2']].max(axis=1)
        df_topn.drop(['frec1', 'frec2', 'porc1', 'porc2'], axis=1, inplace=True) 
        df_topn = df_topn.loc[df_topn.max_freq_of_any_decade>= at_least_in_any_decade].head(n)
        df_topn = df_topn.sort_values('similaridad_semantica', ascending=True).head(n)
        #display(df_topn)
        topn_all_df.append(df_topn)
    
    topn_periodo[par_periodo] = pd.concat(topn_all_df, ignore_index=True)    


#### Intersección promedio
Por el enfoque Procruste para cada par de periodos de 5 años:
¿Cuál es la similaridad semantica promedio (medida de estabilidad del modelo) de los top 250 palabras de las 50 ejecuciones para cada par de periodos comparados con intervalos de confianza del 95% calculados mediante el método bootstrap? 


In [7]:
#def confidence_intervals(data):
#    res = bootstrap((data,), np.mean, confidence_level=0.95)
#    return (res.confidence_interval.low, res.confidence_interval.high)

from scipy.stats import bootstrap
import numpy as np

def confidence_intervals(data):
    """
    Calcula intervalo de confianza con bootstrap de forma robusta
    """
    # Convertir a array numpy y limpiar NaN
    data_clean = np.array(data).copy()
    data_clean = data_clean[~np.isnan(data_clean)]
    
    # Casos especiales donde bootstrap no puede calcularse
    if len(data_clean) < 2:
        # Si hay solo un dato, no hay intervalo de confianza
        if len(data_clean) == 1:
            return (data_clean[0], data_clean[0])
        else:
            return (np.nan, np.nan)
    
    # Si todos los valores son iguales, bootstrap falla
    if np.all(data_clean == data_clean[0]):
        return (data_clean[0], data_clean[0])
    
    try:
        # Configurar bootstrap para evitar problemas con muestras pequeñas
        res = bootstrap(
            (data_clean,), 
            np.mean, 
            confidence_level=0.95,
            random_state=42,  # Para reproducibilidad
            n_resamples=min(1000, len(data_clean) * 100)  # Adaptar a tamaño de muestra
        )
        return (res.confidence_interval.low, res.confidence_interval.high)
    
    except (ValueError, RuntimeError):
        # Fallback: intervalo aproximado usando distribución t
        from scipy import stats
        n = len(data_clean)
        mean = np.mean(data_clean)
        std_err = stats.sem(data_clean)
        
        if n <= 1 or std_err == 0:
            return (mean, mean)
        
        h = std_err * stats.t.ppf((1 + 0.95) / 2., n - 1)
        return (mean - h, mean + h)

In [8]:
topn_periodo_all = topn_periodo.copy() # copiar

In [9]:
#df = topn_periodo[(2009, 2014)].copy()
#data = df['palabra'].value_counts().reset_index()
#data = data[data['count']<2]
#pal_lista = list(data['palabra'].unique())
#pal_lista
# (2009, 2014) = ['tratamiento',  'provincia',  'desastre',  'celula', 'pmo', 'ministerio', 'zona', 'ambito']
#(2014, 2019) = ['nina', 'persona', 'medicina', 'potable', 'ley']

In [10]:
for par_periodo in pares_periodo: 
    display(topn_periodo[par_periodo].shape)
    df = topn_periodo[par_periodo].copy()
    data = df['palabra'].value_counts().reset_index()
    data = data[data['count']>=2]
    pal_lista = list(data['palabra'].unique())
    df = df[df['palabra'].isin(pal_lista)]
    df['vt1'] = df['top10_vecindad_t1'].apply(lambda x: [ peso[0] for peso in eval(x) ])
    df['vt2'] = df['top10_vecindad_t2'].apply(lambda x: [ peso[0] for peso in eval(x) ])
    df['vt1_union'] = df['vt1'].copy()
    df['vt2_union'] = df['vt2'].copy()
    
    #df = df[['par_periodo', 'palabra', 'similaridad_semantica']].groupby(['par_periodo', 'palabra']).agg(['mean', confidence_intervals]).reset_index()
    df = df[['par_periodo', 'palabra', 'similaridad_semantica','vt1','vt2','vt1_union','vt2_union']].groupby(['par_periodo', 'palabra']).agg({
        'similaridad_semantica': ['mean',confidence_intervals],
        'vt1': [ lambda listas: set.intersection(*map(set, listas))],
        'vt2': [ lambda listas: set.intersection(*map(set, listas))],
        'vt1_union': [ lambda listas: set.union(*map(set, listas))],
        'vt2_union': [ lambda listas: set.union(*map(set, listas))]
    }).reset_index()

    #df.sort_values(by='par_periodo', inplace=True)
    df.columns = [
    '_'.join(filter(None, x))
    for x in df.columns.to_flat_index() 
    ]
     	
    df.rename(columns={'vt1_<lambda>': 'vt1_top10', 'vt2_<lambda>': 'vt2_top10','vt1_union_<lambda>': 'vt1_top10_union', 'vt2_union_<lambda>': 'vt2_top10_union'}, inplace=True)
    topn_periodo[par_periodo] = df

    
  
  


(12500, 8)

(12500, 8)

In [11]:
topn_periodo[par_periodo].head(3)

,par_periodo,palabra,similaridad_semantica_mean,similaridad_semantica_confidence_intervals,vt1_top10,vt2_top10,vt1_top10_union,vt2_top10_union
0,"(2014, 2019)",accesibilidad,0.246599,"(0.22559860755138836, 0.2647665261127905)","{discapacitada, consentimiento, movilidad}","{gestion, elemento}","{discapacitada, fertilizacion, actualizacion, consentimiento, judicial, desfibrilador, dato, maternidad, asistido, informado, rehabilitacion, hemocomponent, area, trasplantado, hijo, poblacion, cuyo, fondo, prepagar, politica, automatico, concurrencia, seguridad, movilidad, espacio, garantizar, proteccion, equinoterapia, permanente, presupuesto}","{congenita, elaborado, condicion, calendario, etiqueta, atrofia, examen, licenciado, paliativo, elemento, oftalmologico, visual, tabaco, dato, gratuito, habilitado, ambiental, muscular, digno, producto, similar, gestion, celiaca, ame, fumar, incorporar, espinal, exhibicion, gluten, tecnologico, trabajo, espacio, numerir, poblacion}"
1,"(2014, 2019)",acceso,0.575210,"(0.5596438988046418, 0.5901890391455907)","{potable, formulacion, consumar, comercialicir, solar}","{potable, agua}","{potable, comercialicir, agua, solar, muerte_subita, formulacion, consumar, asistido, deber, regulacion, publico, area, lugar, produccion, gasificado, biomedico, vacunacion, sal, concurrencia, piercing, investigacion, exhibicion, relacionado, garantizar, espacio, radiacion, padecer, epidemiologico, medida}","{potable, aplv, prioritario, acv, departamento, leche, agua, proteina, declarar, edad, necesario, fiebre, privado, cuyo, reproduccion, medicamento, gratuidad, dependencia, calidad, garantizar, punto, proteccion, padecer, provision, minimo, farmacia, apto, universal, vacuna, presupuesto}"
2,"(2014, 2019)",accidente,0.581097,"(0.566722604893749, 0.593561207398835)","{acv, cerebrovascular, vascular, acvo}","{acv, alzheimer, proteina, alergia, cerebrovascular}","{vida, exposicion, caso, acv, mancha, reumatoidea, natural, seguimiento, promover, llamado, recurso, juegos, sida, adquirido, vascular, acvo, radiacion, oporto, minimo, sustancia, vino, rehabilitacion, artriti, transcurso, cerebrovascular, digno, presupuesto}","{aplv, acv, prevenibl, calendario, proteina, leche, anual, violencia, cumplimiento, atrofia, enfermo, fiebre, evaluacion, alzheimer, leyenda, psicologico, artriti, deteccion, cerebrovascular, llamado, malformacion, exigencia, politica, mancha, campana, quimico, hepatitis, espinal, vascular, radiacion, oporto, alergia, vino, publicitario, cirugia, prevencion, regional}"


### Cambios semánticos  

#### Identificar las palabras con cambio semántico importante y moderado

Identificar las palabras que aparecen consistentemente como cambiantes con similitud coseno promedio inferior a 0.5
tal que: 

* Cambio fuerte  (usos muy distintos) : similitud coseno promedio <= 0.3 (umbral 1) 
* Cambio moderado: similitud coseno promedio entre 0.3 y 0.5 (umbral 2)


In [12]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# Convertir datos en DataFrame para visualización
def plot_heatmap(cambios_df, title, top = 10):
    #words, values = zip(*changed_words[:top])  # Tomamos el top 10
    #df = pd.DataFrame({"Palabra": words, "Cambio Semántico": values})
    df = cambios_df.copy()
    plt.figure(figsize=(8, 5))
    sns.heatmap(df.set_index("palabra").T, cmap="Reds", annot=True, fmt=".2f", linewidths=0.5)
    plt.title(title + " Top "+str(top))
    plt.show()



In [13]:
topk = 10
for par_periodo in pares_periodo:

    print("dist > 0.7 (sc <= 0.3) Cambio fuerte (usos muy distintos)")
    periodo_df = topn_periodo[par_periodo]
    umbral_fuerte1 = 0.3
    periodo_df1 =  periodo_df[periodo_df['similaridad_semantica_mean']<=umbral_fuerte1].sort_values('similaridad_semantica_mean', ascending=True).head(topk)
    # Graficar Heatmaps
    #display(periodo_df1)
    #plot_heatmap(periodo_df1[['palabra','similaridad_semantica_mean']], "Cambio semántico  (["+str(par_periodo[0])+"-"+str(par_periodo[1])+") → ["+str(par_periodo[1])+"-"+str(par_periodo[1]+5)+")", top=topk)
    print("TOP 10 Palabras:", periodo_df1['palabra'].unique())
    display(periodo_df1) 
    print("dist ~ 0.5 (sc ~ 0.5)  Cambio moderado")

    umbral_fuerte2 = 0.5
    periodo_df1 =  periodo_df[(periodo_df['similaridad_semantica_mean']<=umbral_fuerte2) & (periodo_df['similaridad_semantica_mean']>umbral_fuerte1)].sort_values('similaridad_semantica_mean', ascending=True).head(topk)
    # Graficar Heatmaps
    #plot_heatmap(periodo_df1[['palabra','similaridad_semantica_mean']], "Cambio semántico  (["+str(par_periodo[0])+"-"+str(par_periodo[1])+") → ["+str(par_periodo[1])+"-"+str(par_periodo[1]+5)+")", top=topk)
    print("TOP 10 Palabras:", periodo_df1['palabra'].unique())
    display(periodo_df1)    


dist > 0.7 (sc <= 0.3) Cambio fuerte (usos muy distintos)
TOP 10 Palabras: ['legal' 'declaracion']


,par_periodo,palabra,similaridad_semantica_mean,similaridad_semantica_confidence_intervals,vt1_top10,vt2_top10,vt1_top10_union,vt2_top10_union
186,"(2009, 2014)",legal,0.230416,"(0.21897246163264292, 0.24237258551177132)","{musicoterapia, colaborador, auxiliar, obstetricia, norma, odontologia, ejercicio, loteria}","{monodroga, droga}","{musicoterapia, expendio, capitulo, colaborador, eliminacion, obstetricia, norma, odontologia, parrafo, ocupacional, medicina, juegos, cosmetologo, cosmetologia, auxiliar, complementario, terapeuta, odontologico, apuesta, ejercicio, loteria, gravamen, fitoterapicos}","{misoprostol, comprendido, calendario, proteina, implante, estudio, droga, elemento, oftalmologico, fabricacion, empleo, formulacion, fisico, recoleccion, alergena, vencido, procreacion, domiciliario, planta, ftalato, vegetal, instrumento, fitoterapicos, menor, sufrir, produccion, exigencia, aptitud, monodroga, tipo, caso, efecto, realizacion, medicamento, embarazo, medicamentosa, laser, mercurio, garantizar, fetal, cirugia, vacuna, sol}"
83,"(2009, 2014)",declaracion,0.241996,"(0.2249657157631225, 0.25862471168390483)","{natural, chaco, rioja, jujuy, valle}","{valor, medicina, marco, prepagar}","{desastre, departamento, pedro, corriente, natural, chaco, rioja, declarar, atrofia, plazo, enfermeria, valle, espinal, accion, emergencia, vital, ceniza, muerte, dengue, ochenta, mendoza, epidemiologico, alergia, jujuy, tecnico, volcanica, rio}","{receta, entidad, exigencia, comision, prepagar, familia, listado, judicial, afiliado, marco, requisito, medicina, publicar, valor, tecnologico, trabajo, maternidad, pami, impuesto, suspension, transcurso, inssjp, lectivo}"


dist ~ 0.5 (sc ~ 0.5)  Cambio moderado
TOP 10 Palabras: ['respectivamente' 'empleo' 'materia' 'el' 'tipo' 'ingreso' 'patologia'
 'territorio' 'pre' 'acompanante']


,par_periodo,palabra,similaridad_semantica_mean,similaridad_semantica_confidence_intervals,vt1_top10,vt2_top10,vt1_top10_union,vt2_top10_union
294,"(2009, 2014)",respectivamente,0.326138,"(0.31031950910206957, 0.34117638271974926)","{ajuste, potable, gravis, semana, gasificado}","{autoridad, gasificado}","{potable, gravis, osteoporosis, rango, chaga, bis, agua, semana, escala, anual, equivalente, remuneracion, mastectomia, monto, ajuste, top, ayuda, miastenia, gratuito, gasificado, acceder}","{premio, potable, sexo, pensionado, judicial, consentimiento, nutricion, congreso, relacion, requisito, donante, monto, subsidio, valor, exencion, composicion, mayo, informado, autoridad, regulacion, habilitado, gasificado, el, pension, jubilacion, obligacion, monohidratado, tecnico, apto, suspension, discapacidad}"
120,"(2009, 2014)",empleo,0.329805,"(0.3143491522050961, 0.34312720552187775)","{jefe, familia, observatorio, hogar, comunitario, monto}","{monodroga, sustancia, contributivo, subsistema, ftalato, instrumento, elemento}","{rural, jefe, familia, rango, pacto, republica, consejo, equivalente, hogar, formacion, remuneracion, monto, revestir, ceniza, erupcion, observatorio, aprobacion, nivel, volcanica, comunitario, autorizar, fuerza}","{misoprostol, monodroga, formacion, elemento, laser, mercurio, alergena, sustancia, contributivo, asignacion, subsistema, ftalato, instrumento}"
197,"(2009, 2014)",materia,0.347583,"(0.33340934614428197, 0.36327145129281574)","{adopcion, postraumatico, fortalecimiento, basica, estr, presupuesto, primaria}","{muerte, politica, congreso, vacunacion, cumplimiento}","{adopcion, fortalecer, postraumatico, fortalecimiento, hipoacusia, red, prostata, objetivo, basica, financiero, delito, cerebrovascular, estr, presupuesto, primaria}","{politica, consentimiento, prevenibl, congreso, vacunacion, violencia, glifosato, cumplimiento, fiebre, economico, seguridad, formulacion, laboratorio, trabajo, recoleccion, muerte, garantizar, obligacion, domiciliario, vencido, area, produccion, agropecuario}"
113,"(2009, 2014)",el,0.367072,"(0.347573663656707, 0.3829243480321881)","{identificar, rioja, septiembre, regional}","{premio, doctor, septiembre}","{prioritario, quistico, endemico, ambulancia, rioja, comercializado, infanto, hidrogenado, autorizar, fibromialgia, infeccion, argentina, autonoma, juvenil, mundial, aceite, identificar, ciudad, febrero, septiembre, regional}","{premio, nacimiento, consentimiento, creatin, acreditar, octubre, glifosato, doctor, distribucion, edad, clinico, fabricacion, formulacion, respectivamente, informado, importacion, institucion, produccion, lectivo, menor, recetado, monodroga, expendio, prohibir, septiembre, julio, inter, monohidratado, mercurio, control, lavado}"
327,"(2009, 2014)",tipo,0.369607,"(0.3536837503623812, 0.38601317491137227)","{virus, gripe, papiloma, influenza, aparicion, causado, h1n1}","{ensenanza, cualquiera}","{gripe, varicela, creasar, hpv, vacunacion, asistencial, neumococo, hepatitis, h1n1, virus, avance, cobro, incremento, papiloma, influenza, reconocer, pandemia, territorio, aparicion, causado}","{frio, legal, aplv, perforacion, organo, proteina, tabaquismo, distribucion, droga, fabricacion, deportivo, importacion, odontologico, practica, planta, cadena, instrumento, sufrir, aptitud, prohibir, nomenclador, cualquiera, tatuaj, piercing, medicamentosa, incorporar, integrante, mercurio, fetal, vacuna, recurso, ensenanza}"
176,"(2009, 2014)",ingreso,0.371601,"(0.3565678796123401, 0.3883873436702735)","{geriatrico, fincini, ganancia, basico, ciudadano, ninez}","{categoria, ninez}","{sexo, correspondiente, conexa, ordenado, ganancia, trabajador, distribucion, prenupcial, fisico, fincini, ciudadano, reconocimiento, diabetes, ciento, fondo, geriatrico, monodroga, pension, asistencial, consultoria, adiccion, jubilacion, prohibicion, dieciseis, revestir, anses, desocupado, mental, medicacion, certificado, basico, militar, ninez}","{federal, loteria, politica, conexa, 

dist > 0.7 (sc <= 0.3) Cambio fuerte (usos muy distintos)
TOP 10 Palabras: ['dispositivo' 'adecuado' 'accesibilidad']


,par_periodo,palabra,similaridad_semantica_mean,similaridad_semantica_confidence_intervals,vt1_top10,vt2_top10,vt1_top10_union,vt2_top10_union
96,"(2014, 2019)",dispositivo,0.222339,"(0.20644338379652574, 0.23954531318495303)","{nocivo, dato, similar, electronico}","{minimo, oftalmologico, tecnologico, muerte_subita}","{correspondiente, geneticamente, comprendido, listado, infanto, requisito, necesario, demas, leches, dato, historia, medica, bullos, similar, receta, efecto, osteoporosis, campana, medicamentosa, rotulo, capacidad, digital, incorporar, organismo, nocivo, implementacion, consumidor, cancer, secundario, administracion, electronico, transcurso, ciudadano, transgenico}","{privado, potable, comprendido, cartel, automatico, concurrencia, enfermo, examen, desfibrilador, instalacion, aplicacion, calidad, muerte_subita, oftalmologico, adolescente, tecnologico, pobreza, escolar, educativo, radiacion, minimo, gratuito, cuidado, lugar, cronico}"
5,"(2014, 2019)",adecuado,0.241482,"(0.22508851871368224, 0.25475585614389196)",{rotulacion},"{geriatrico, tumor, manejo}","{especialidad, frio, potable, alcohol, elaborado, circuito, capitulo, rotulacion, complemento, television, jugo, composicion, informado, modificatoria, alimentacion, cerrado, alimentario, planta, contenido, cadena, gasificado, codigo, profesion, braille, medicinal, camara, seguridad, porcentaje}","{personal, congreso, paciente, pais, manejo, paliativo, elemento, dato, alzheimer, modificatoria, funcion, procreacion, domiciliario, diabetes, cerebrovascular, hijo, vida, ejecutivo, agente, geriatrico, nomenclador, ministerio, mujer, mama, multiple, tumor, situacion, financiero}"
0,"(2014, 2019)",accesibilidad,0.246599,"(0.22559860755138836, 0.2647665261127905)","{discapacitada, consentimiento, movilidad}","{gestion, elemento}","{discapacitada, fertilizacion, actualizacion, consentimiento, judicial, desfibrilador, dato, maternidad, asistido, informado, rehabilitacion, hemocomponent, area, trasplantado, hijo, poblacion, cuyo, fondo, prepagar, politica, automatico, concurrencia, seguridad, movilidad, espacio, garantizar, proteccion, equinoterapia, permanente, presupuesto}","{congenita, elaborado, condicion, calendario, etiqueta, atrofia, examen, licenciado, paliativo, elemento, oftalmologico, visual, tabaco, dato, gratuito, habilitado, ambiental, muscular, digno, producto, similar, gestion, celiaca, ame, fumar, incorporar, espinal, exhibicion, gluten, tecnologico, trabajo, espacio, numerir, poblacion}"


dist ~ 0.5 (sc ~ 0.5)  Cambio moderado
TOP 10 Palabras: ['forma' 'tipo' 'domiciliario' 'comunitario' 'necesario' 'educacion'
 'pais' 'manejo' 'obligacion' 'sustancia']


,par_periodo,palabra,similaridad_semantica_mean,similaridad_semantica_confidence_intervals,vt1_top10,vt2_top10,vt1_top10_union,vt2_top10_union
143,"(2014, 2019)",forma,0.302599,"(0.28734555124627215, 0.31576821693124263)","{transmision, proteina}","{correspondiente, organizacion, actualizacion}","{fertilizacion, elaborado, derogacion, anonimo, complemento, sexual, prevenibl, bis, proteina, agua, tabaquismo, vih, examen, almacenamiento, solar, prenupcial, derivado, tecnica, oximetria, consumar, leyenda, educativo, gratuito, fitoterapicos, humano, biomedico, comida, celiaca, cartel, cualquiera, reproduccion, sal, transmision, adiccion, investigacion, exhibicion, pulso, relacionado, apto, nivel, ensenanza, utilizacion}","{correspondiente, familia, actualizacion, condicion, comite, semana, bis, indice, droga, primario, valor, fisico, respectivamente, denominado, modificatoria, funcion, domiciliario, muscular, digno, hijo, establecido, corporal, seguro, organizacion, nomenclador, celiaca, cualquiera, automatico, espinal, traves, gluten, ter, otorgar, mundial, trabajo, relacionado, modelo, punto, secundario, sindrome, publicitario, medida, perjudicial, septiembre, regional}"
317,"(2014, 2019)",tipo,0.316068,"(0.29948764875959016, 0.33379149222781307)","{ensenanza, cualquiera}","{diabetes, braille, droga}","{frio, legal, aplv, perforacion, organo, proteina, tabaquismo, distribucion, droga, fabricacion, deportivo, importacion, odontologico, practica, planta, cadena, instrumento, sufrir, aptitud, prohibir, nomenclador, cualquiera, tatuaj, piercing, medicamentosa, incorporar, integrante, mercurio, fetal, vacuna, recurso, ensenanza}","{prepagar, pensionado, judicial, educacion, animal, diabetes, capacidad, braille, droga, identificacion, tecnica, visual, fisico, trabajo, secundario, extension, sustitucion, certificado, domiciliario, trasplantado, discapacidad, afectar}"
100,"(2014, 2019)",domiciliario,0.333025,"(0.31695472356673887, 0.3514712556150475)","{vencido, recoleccion}","{ter, educacion, financiero}","{frio, entidad, comision, pensionado, comercialicir, tecnologia, paliativo, droga, autonoma, anmat, recoleccion, dato, maternidad, dengue, materia, deber, vencido, inssjp, planta, cadena, digno, vegetal, fitoterapicos, seguro, jubilado, consejo, fortalecimiento, medicamento, braille, medicinal, tecnologico, muerte, pami, financiero, resolucion, autorizar}","{especial, obstetricia, formacion, paliativo, primario, terapeutico, fisico, respectivamente, acompanante, extension, actividad, forma, rehabilitacion, adecuado, cuidado, animal, diabetes, trasplantado, hijo, fondo, organizacion, tipo, educacion, cualquiera, ter, tecnologico, trabajo, financiero, tumor, tecnico, ejercicio, certificado, equinoterapia}"
62,"(2014, 2019)",comunitario,0.339957,"(0.3247852523503728, 0.3536434896428815)","{accion, marzo}","{agente, escolar, organizacion}","{desarrollo, comision, nacion, estatal, familia, listado, prestacion, afiliado, marco, octubre, violencia, enfermo, licenciado, paliativo, marzo, accion, valor, maternidad, mayo, mes, ambito, febrero, digno, primaria, prepagar, fortalecimiento, natural, julio, proceso, calidad, adiccion, organismo, victima, medicina, cancer, mental, financiero, declaracion, enero, ninez}","{presupuesto, agente, cuyo, corporal, organizacion, especialidad, capacitacion, registro, fortalecimiento, comida, trabajador, obstetricia, estetico, norma, tecnologia, objeto, escolar, punto, tumor, reconocimiento, modificatoria, situacion, regulacion, ejercicio, complejidad, grasa, establecido}"
221,"(2014, 2019)",necesario,0.347956,"(0.3280727319221616, 0.3699679241238856)","{desfibrilador, dispositivo}","{principio, precio}","{sexo, diagnostico, masivo, semana, desfibrilador, prenupcial, muerte_subita, dispositivo, celiaco, precio, deportivo, practica, medica, loteria, aptitud, tipo, osteoporosis, comida, celiaca, realizacion, automatico, concurrencia, externo, piercing, incorporar, mama, general, fetal, cirugia, apto, existir,

In [14]:
# GUARDAR OBJETO
with open(modelo_dir+'/top250_periodos_procrustes_df.pkl', 'wb') as file: 
    pickle.dump(topn_periodo,file)

Palabras comunes entre los pares de períodos

In [15]:
# palabra común en entre períodos
palabra_comun = set(topn_periodo[(2009, 2014)]['palabra'].unique()) & set(topn_periodo[(2014, 2019)]['palabra'].unique())
print("Cantidad de palabras en común:",len(palabra_comun))

Cantidad de palabras en común: 276


Top 10

In [16]:
# Período (2009, 2014)
df = topn_periodo[(2009, 2014)]
df = df[df['palabra'].isin(palabra_comun)]
df = df[['palabra','similaridad_semantica_mean','similaridad_semantica_confidence_intervals','vt1_top10','vt2_top10']]
df.columns = ['palabra','ss_mean_2009_2014','ss_ci_2009_2014','vt1_top10_2009_2014','vt2_top10_2009_2014']
# Período (2014, 2019)
df2 = topn_periodo[(2014, 2019)]
df2 = df2[df2['palabra'].isin(palabra_comun)]
df2 = df2[['palabra','similaridad_semantica_mean','similaridad_semantica_confidence_intervals','vt1_top10','vt2_top10']]
df2.columns = ['palabra','ss_mean_2014_2019','ss_ci_2014_2019','vt1_top10_2014_2019','vt2_top10_2014_2019']
# Unión
df = df.merge(df2, how='inner', on='palabra')
df = df[['palabra', 'ss_mean_2009_2014', 'ss_mean_2014_2019', 'ss_ci_2009_2014','ss_ci_2014_2019','vt1_top10_2009_2014','vt2_top10_2009_2014', 'vt1_top10_2014_2019','vt2_top10_2014_2019']]
df['peso'] = (df['ss_mean_2009_2014'] + df['ss_mean_2014_2019'] )/2
print("Top 10 de palabras comunes con Fuerte cambio semántco entre Período [2009, 2014) - > [2014, 2019)  ")
print("Palabras:",df[df['peso']<=0.3].sort_values('peso').head(10)['palabra'].unique() )
display(df[df['peso']<=0.3].sort_values('peso').head(10))
print("Top 10 de palabras comunes con moderado cambio semántco entre Período [2009, 2014) - > [2014, 2019)  ")
print("Palabras:",df[(df['peso']>0.3) & (df['peso']<=0.5)].sort_values('peso').head(10)['palabra'].unique() )
display(df[(df['peso']>0.3) & (df['peso']<=0.5)].sort_values('peso').head(10))



Top 10 de palabras comunes con Fuerte cambio semántco entre Período [2009, 2014) - > [2014, 2019)  
Palabras: []


,palabra,ss_mean_2009_2014,ss_mean_2014_2019,ss_ci_2009_2014,ss_ci_2014_2019,vt1_top10_2009_2014,vt2_top10_2009_2014,vt1_top10_2014_2019,vt2_top10_2014_2019,peso


Top 10 de palabras comunes con moderado cambio semántco entre Período [2009, 2014) - > [2014, 2019)  
Palabras: ['tipo' 'legal' 'ingreso' 'materia' 'forma' 'argentina' 'comunitario'
 'respectivamente' 'domiciliario' 'derivado']


,palabra,ss_mean_2009_2014,ss_mean_2014_2019,ss_ci_2009_2014,ss_ci_2014_2019,vt1_top10_2009_2014,vt2_top10_2009_2014,vt1_top10_2014_2019,vt2_top10_2014_2019,peso
260,tipo,0.369607,0.316068,"(0.3536837503623812, 0.38601317491137227)","(0.29948764875959016, 0.33379149222781307)","{virus, gripe, papiloma, influenza, aparicion, causado, h1n1}","{ensenanza, cualquiera}","{ensenanza, cualquiera}","{diabetes, braille, droga}",0.342838
149,legal,0.230416,0.531215,"(0.21897246163264292, 0.24237258551177132)","(0.5135781148578317, 0.5453438046438103)","{musicoterapia, colaborador, auxiliar, obstetricia, norma, odontologia, ejercicio, loteria}","{monodroga, droga}","{monodroga, droga}",{obstetricia},0.380816
141,ingreso,0.371601,0.397244,"(0.3565678796123401, 0.3883873436702735)","(0.3843759449067928, 0.40972211038371514)","{geriatrico, fincini, ganancia, basico, ciudadano, ninez}","{categoria, ninez}","{categoria, ninez}","{otorgar, jubilado, beneficiario, pensionado, fortalecimiento, primario}",0.384422
159,materia,0.347583,0.422933,"(0.33340934614428197, 0.36327145129281574)","(0.40989486375773637, 0.43562222976790543)","{adopcion, postraumatico, fortalecimiento, basica, estr, presupuesto, primaria}","{muerte, politica, congreso, vacunacion, cumplimiento}","{muerte, politica, congreso, vacunacion, cumplimiento}","{vih, prorrogable, zona, desastre, infeccion}",0.385258
120,forma,0.484979,0.302599,"(0.46719765405327235, 0.5014406845294406)","(0.28734555124627215, 0.31576821693124263)","{chagas, indirecto, realizacion, directo, transmision}","{transmision, proteina}","{transmision, proteina}","{correspondiente, organizacion, actualizacion}",0.393789
22,argentina,0.408254,0.382985,"(0.3951203781083962, 0.4197302601418776)","(0.36657267596987847, 0.400737804981364)","{limitacion, cigarrillo, prioritario, republica, bioetica, chagas, comercializado}","{republica, comision, natural}","{republica, comision, natural}","{exigencia, evaluacion, republica, pobreza, tecnologia, desfibrilador}",0.395619
50,comunitario,0.460755,0.339957,"(0.44579246758973323, 0.47530858435176465)","(0.3247852523503728, 0.3536434896428815)","{radicacion, revestir, voluntario, hogar, direccion, militar, sanitaria}","{accion, marzo}","{accion, marzo}","{agente, escolar, organizacion}",0.400356
236,respectivamente,0.326138,0.479367,"(0.31031950910206957, 0.34117638271974926)","(0.46187416410691934, 0.4959425002968709)","{ajuste, potable, gravis, semana, gasificado}","{autoridad, gasificado}","{autoridad, gasificado}","{fisico, publicidad}",0.402753
82,domiciliario,0.495901,0.333025,"(0.4786382583568243, 0.5093883782018698)","(0.31695472356673887, 0.3514712556150475)","{recoleccion, mercurio, civil, odontologico, partos, vencido, capacidad, instrumento}","{vencido, recoleccion}","{vencido, recoleccion}","{ter, educacion, financiero}",0.414463
71,derivado,0.459340,0.390248,"(0.44670550331782066, 0.4710231172662997)","(0.3726978734779245, 0.4054997895619037)","{gluten, envase, alergena, geneticamente}","{exposicion, geneticamente, modificado}","{exposicion, geneticamente, modificado}","{planta, medicinal, organismo}",0.424794


In [17]:
# GUARDAR OBJETO
with open(modelo_dir+'/palComun_cambioSem_procrustes_df.pkl', 'wb') as file: 
    pickle.dump(df,file)

### Chat GPT

#### Breve historia del sistema argentino de salud entre 2009 a 2023

Destacando los principales cambios, políticas y desafíos en ese período:

**2009-2015: Ampliación de cobertura y enfoque en salud pública**
* Programa Nacer (iniciado antes) se transformó en Programa SUMAR (2012), ampliando la cobertura sanitaria gratuita a población sin obra social, especialmente niños, embarazadas y adultos hasta 64 años.

* Se fortalecieron las políticas de atención primaria, la distribución gratuita de medicamentos esenciales y el control de enfermedades crónicas.

* Ley Nacional de Salud Mental (2010): marcó un cambio paradigmático hacia una atención centrada en la comunidad, con foco en derechos humanos.

* Inversión en infraestructura y en redes de servicios provinciales, aunque con desigualdades persistentes entre regiones.

**2016-2019: Intento de reformas estructurales y retrocesos por la crisis**

* Lanzamiento de la Cobertura Universal de Salud (CUS) en 2016, con el objetivo de mejorar el acceso de personas sin cobertura formal; sin embargo, tuvo una implementación limitada.

* La crisis económica (inflación, devaluación, recorte del gasto) afectó recursos para salud pública y programas nacionales.

* Tensiones con obras sociales y prepagas por regulaciones, deudas y aranceles.

* En 2018, debate parlamentario por la legalización del aborto generó una movilización social histórica, aunque fue rechazado por el Senado.

**2020-2023: Pandemia, legalización del aborto y digitalización**
* COVID-19 (2020-2021) puso al sistema bajo gran presión: se ampliaron camas, respiradores, se contrató personal y se distribuyeron recursos federales.

* Se desarrolló un plan nacional de vacunación masivo y gratuito, con campañas públicas y centros de vacunación en todo el país.

* En diciembre de 2020 se aprobó la Ley 27.610 de Interrupción Voluntaria del Embarazo, garantizando el derecho a acceder a un aborto legal y seguro hasta la semana 14.

* Se fortaleció la telemedicina y se impulsó la digitalización de historiales clínicos, aunque de manera desigual según la provincia.

* En 2022-2023, el sistema volvió a enfrentar restricciones presupuestarias por la crisis económica, con reclamos de trabajadores de salud, atraso en pagos a proveedores y conflicto con prepagas por aumentos de cuotas y regulación estatal.

#### Conclusión
El sistema de salud argentino entre 2009 y 2023 mostró avances importantes en derechos y cobertura, especialmente con programas públicos y leyes clave (salud mental, aborto). Pero también atravesó crisis económicas y desafíos estructurales que afectaron la calidad y equidad del acceso, manteniéndose como un sistema fragmentado y desigual, con fuerte dependencia de decisiones políticas nacionales y provinciales.

# Anexo

#### Como detectar cambios liguistico con chatgpt

Detectar cambios lingüísticos con ChatGPT u otros modelos de lenguaje puede hacerse de varias formas, dependiendo de qué tipo de cambio te interesa (léxico, semántico, sintáctico, etc.).

* Comparación sincrónica: simular diferentes “épocas” pidiéndole al modelo que hable como si fuera de cierto año:

🗣️ Prompt:
“¿Cómo se usaba la palabra ‘cuidar’ en el contexto de salud pública en Argentina en 2010?”
vs.
“¿Cómo se usa la palabra ‘cuidar’ en el contexto de salud pública en Argentina en 2023?”

* Análisis con embeddings + ChatGPT

Asistente para interpretar cambios de significado basados en embeddings entrenados por año. Ejemplo:
Entrenás vectores (word embeddings) para distintos años. Medís distancia o cambio de contexto de ciertas palabras clave. Usás ChatGPT para ayudarte a interpretar esos cambios

🔍 Prompt:
“En 2010, la palabra ‘prevención’ estaba más cerca de ‘vacuna’, ‘campaña’, ‘dengue’. En 2023, está más cerca de ‘cuidado’, ‘género’, ‘territorio’. ¿Qué tipo de cambio semántico puede estar ocurriendo?”